# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [1]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder


# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [2]:
# Load the dataset from Phase 2 (update the path as needed)
DATA_PATH = r"D:\Churn Prediction And Analysis Project\Cell2Cell Data\cell2celltrain.csv"

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")

df.head().T

Loaded dataset: 51047 rows x 58 columns


,0,1,2,3,4
CustomerID,3000002,3000010,3000014,3000022,3000026
Churn,Yes,Yes,No,No,Yes
MonthlyRevenue,24.0,16.99,38.0,82.28,17.14
MonthlyMinutes,219.0,10.0,8.0,1312.0,0.0
TotalRecurringCharge,22.0,17.0,38.0,75.0,17.0
DirectorAssistedCalls,0.25,0.0,0.0,1.24,0.0
OverageMinutes,0.0,0.0,0.0,0.0,0.0
RoamingCalls,0.0,0.0,0.0,0.0,0.0
PercChangeMinutes,-157.0,-4.0,-2.0,157.0,0.0
PercChangeRevenues,-19.0,0.0,0.0,8.1,-0.2


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [3]:
# TODO: Select the relevant columns and rows for your analysis.

columns_to_drop = [
    'CustomerID',
    'DroppedBlockedCalls',
    'NotNewCellphoneUser'
]

drop_reason = {
    'CustomerID': 'Unique identifier with no predictive value.',
    'DroppedBlockedCalls': 'Redundant feature derived from DroppedCalls and BlockedCalls.',
    'NotNewCellphoneUser': 'Redundant inverse of NewCellphoneUser.'
}

df_selected = df.drop(
    columns=columns_to_drop,
    errors='ignore'
)

print(f"Original Shape: {df.shape}")
print(f"Selected Shape: {df_selected.shape}")
print(f"\nNumber of Dropped Features: {len(columns_to_drop)}")

print("\nDropped Features and Reasons:")
for col in columns_to_drop:
    print(f"- {col}: {drop_reason[col]}")

Original Shape: (51047, 58)
Selected Shape: (51047, 55)

Number of Dropped Features: 3

Dropped Features and Reasons:
- CustomerID: Unique identifier with no predictive value.
- DroppedBlockedCalls: Redundant feature derived from DroppedCalls and BlockedCalls.
- NotNewCellphoneUser: Redundant inverse of NewCellphoneUser.


In [4]:
df_selected.head().T

,0,1,2,3,4
Churn,Yes,Yes,No,No,Yes
MonthlyRevenue,24.0,16.99,38.0,82.28,17.14
MonthlyMinutes,219.0,10.0,8.0,1312.0,0.0
TotalRecurringCharge,22.0,17.0,38.0,75.0,17.0
DirectorAssistedCalls,0.25,0.0,0.0,1.24,0.0
OverageMinutes,0.0,0.0,0.0,0.0,0.0
RoamingCalls,0.0,0.0,0.0,0.0,0.0
PercChangeMinutes,-157.0,-4.0,-2.0,157.0,0.0
PercChangeRevenues,-19.0,0.0,0.0,8.1,-0.2
DroppedCalls,0.7,0.3,0.0,52.0,0.0


In [5]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

# df_selected = df_selected[df_selected['some_column'].notna()]
# print(f"Shape after row selection: {df_selected.shape}")

---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [6]:
# ==========================================
# Phase 3 - Task 2: Clean Data
# ==========================================

# Create a clean copy
df_clean = df_selected.copy()

# ------------------------------------------
# 1. Check Missing Values
# ------------------------------------------

missing_count = df_clean.isnull().sum().sort_values(ascending=False)
missing_percent = (df_clean.isnull().mean() * 100).sort_values(ascending=False)

missing_summary = (
    pd.DataFrame({
        "Missing Count": missing_count,
        "Missing Percentage (%)": missing_percent
    })
    .query("`Missing Count` > 0")
)

print("Missing Values Summary:")
print(missing_summary)

# ------------------------------------------
# 2. Remove Columns with Excessive Missing Values
# ------------------------------------------

missing_threshold = 50  # percentage

cols_to_drop = missing_percent[missing_percent > missing_threshold].index.tolist()

if cols_to_drop:
    df_clean.drop(columns=cols_to_drop, inplace=True)
    print("\nDropped Columns:")
    print(cols_to_drop)
else:
    print("\nNo columns were dropped (no column exceeded 50% missing values).")

# ------------------------------------------
# 3. Handle Missing Values
# ------------------------------------------

# Numerical features → Median
num_cols = df_clean.select_dtypes(include="number").columns

for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical features → Mode
cat_cols = df_clean.select_dtypes(include="object").columns

for col in cat_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# ------------------------------------------
# 4. Validate Missing Values
# ------------------------------------------

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())

Missing Values Summary:
                       Missing Count  Missing Percentage (%)
AgeHH2                           909                1.780712
AgeHH1                           909                1.780712
PercChangeMinutes                367                0.718945
PercChangeRevenues               367                0.718945
MonthlyMinutes                   156                0.305601
TotalRecurringCharge             156                0.305601
DirectorAssistedCalls            156                0.305601
OverageMinutes                   156                0.305601
RoamingCalls                     156                0.305601
MonthlyRevenue                   156                0.305601
ServiceArea                       24                0.047015
CurrentEquipmentDays               1                0.001959
HandsetModels                      1                0.001959
Handsets                           1                0.001959

No columns were dropped (no column exceeded 50% missing valu

In [7]:
# Remove duplicate records.

before = len(df_clean)
df_clean = df_clean.drop_duplicates()
after = len(df_clean)
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

Removed 0 duplicate rows. Remaining: 51047 rows.


In [8]:
# Correct Invalid Values (Negative)

invalid_cols = [
'MonthlyRevenue',
'TotalRecurringCharge',
'CurrentEquipmentDays'
]
for col in invalid_cols:
    print(col, ":", (df_clean[col] < 0).sum())

# Replace Negative Values
for col in invalid_cols:
    df_clean[col] = df_clean[col].clip(lower=0)

# Validation
print("\nRemaining Invalid Values:")

for col in invalid_cols:
    print(col, ":", (df_clean[col] < 0).sum())

MonthlyRevenue : 3
TotalRecurringCharge : 8
CurrentEquipmentDays : 76

Remaining Invalid Values:
MonthlyRevenue : 0
TotalRecurringCharge : 0
CurrentEquipmentDays : 0


In [9]:
# ------------------------------------------
# Handle Outliers using IQR Winsorization
# ------------------------------------------

# Features selected for outlier treatment
outlier_cols = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "TotalRecurringCharge",
    "DirectorAssistedCalls",
    "OverageMinutes",
    "RoamingCalls",
    "PercChangeMinutes",
    "PercChangeRevenues",
    "DroppedCalls",
    "BlockedCalls",
    "UnansweredCalls",
    "CustomerCareCalls",
    "ReceivedCalls",
    "OutboundCalls",
    "InboundCalls",
    "PeakCallsInOut",
    "OffPeakCallsInOut",
    "CurrentEquipmentDays"
]

# Store summary
outlier_summary = []

# Apply IQR Clipping
for col in outlier_cols:

    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()

    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

    after = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers Before": before,
        "Outliers After": after
    })

# Display Summary
outlier_summary = pd.DataFrame(outlier_summary)

print("Outlier Treatment Summary:\n")
print(outlier_summary)

Outlier Treatment Summary:

                  Feature  Outliers Before  Outliers After
0          MonthlyRevenue             3009               0
1          MonthlyMinutes             2588               0
2    TotalRecurringCharge              824               0
3   DirectorAssistedCalls             5530               0
4          OverageMinutes             5980               0
5            RoamingCalls            10070               0
6       PercChangeMinutes             6926               0
7      PercChangeRevenues            13471               0
8            DroppedCalls             3712               0
9            BlockedCalls             5517               0
10        UnansweredCalls             3630               0
11      CustomerCareCalls             6721               0
12          ReceivedCalls             3641               0
13          OutboundCalls             3342               0
14           InboundCalls             4973               0
15         PeakCallsInOut   

In [10]:
# ------------------------------------------
# Final Validation
# ------------------------------------------

print("Final Dataset Shape:")
print(df_clean.shape)

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum().sum())

print("\nDuplicate Records:")
print(df_clean.duplicated().sum())

print("\nNegative Values Check:")

negative_cols = [
    "MonthlyRevenue",
    "TotalRecurringCharge",
    "CurrentEquipmentDays"
]

for col in negative_cols:
    negatives = (df_clean[col] < 0).sum()
    print(f"{col}: {negatives}")

print("\nTask 2 Completed Successfully.")

Final Dataset Shape:
(51047, 55)

Remaining Missing Values:
0

Duplicate Records:
0

Negative Values Check:
MonthlyRevenue: 0
TotalRecurringCharge: 0
CurrentEquipmentDays: 0

Task 2 Completed Successfully.


In [11]:
"""
Cleaning Summary:

Missing values were identified in several numerical and categorical features. Since all missing percentages were relatively low, no columns were removed. Numerical missing values were imputed using the median to reduce the influence of skewed distributions and outliers, while categorical missing values were filled using the mode.

The dataset was checked for duplicate records, and no duplicate rows were found.

Invalid negative values were detected in MonthlyRevenue, TotalRecurringCharge, and CurrentEquipmentDays. These values were considered inconsistent with the business context and were corrected by replacing them with zero.

Outliers were identified using the Interquartile Range (IQR) method. Rather than treating every numerical feature, IQR-based winsorization (clipping) was applied only to selected continuous variables where extreme values were likely to influence model performance. Zero-inflated count variables and ordinal features were excluded from outlier treatment because their extreme values represent valid customer behaviour or categorical levels rather than data quality issues.

Final validation confirmed that the cleaned dataset contains no missing values, no duplicate records, no invalid negative values, and no remaining outliers in the treated features.

The dataset is now clean, consistent, and ready for the feature engineering stage.
"""

'\nCleaning Summary:\n\nMissing values were identified in several numerical and categorical features. Since all missing percentages were relatively low, no columns were removed. Numerical missing values were imputed using the median to reduce the influence of skewed distributions and outliers, while categorical missing values were filled using the mode.\n\nThe dataset was checked for duplicate records, and no duplicate rows were found.\n\nInvalid negative values were detected in MonthlyRevenue, TotalRecurringCharge, and CurrentEquipmentDays. These values were considered inconsistent with the business context and were corrected by replacing them with zero.\n\nOutliers were identified using the Interquartile Range (IQR) method. Rather than treating every numerical feature, IQR-based winsorization (clipping) was applied only to selected continuous variables where extreme values were likely to influence model performance. Zero-inflated count variables and ordinal features were excluded fro

---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [12]:
# ------------------------------------------
# Feature 1: Extract Market from ServiceArea
# ------------------------------------------

# Extract Market Code
df_clean["Market"] = (
    df_clean["ServiceArea"]
    .fillna("Unknown")
    .str[:3]
)

# Remove original column
df_clean.drop(columns="ServiceArea", inplace=True)

# Validation
print("Unique Markets:", df_clean["Market"].nunique())

print("\nTop 10 Markets:")
print(df_clean["Market"].value_counts().head(10))

Unique Markets: 57

Top 10 Markets:
Market
NYC    5694
LAX    3395
SFR    2701
APC    2416
DAL    2393
SAN    2212
CHI    2015
FLN    1906
MIA    1893
ATL    1802
Name: count, dtype: int64


In [13]:
# ------------------------------------------
# Feature 2: Customer Usage Features
# ------------------------------------------

# Total number of calls
df_clean["TotalCalls"] = (
    df_clean["ReceivedCalls"] +
    df_clean["OutboundCalls"] +
    df_clean["InboundCalls"]
)

# Revenue generated per minute
df_clean["RevenuePerMinute"] = np.where(
    df_clean["MonthlyMinutes"] > 0,
    df_clean["MonthlyRevenue"] / df_clean["MonthlyMinutes"],
    0
)

# Average monthly usage per handset
df_clean["UsagePerDevice"] = np.where(
    df_clean["Handsets"] > 0,
    df_clean["MonthlyMinutes"] / df_clean["Handsets"],
    0
)

# ------------------------------------------
# Validation
# ------------------------------------------

new_features = [
    "TotalCalls",
    "RevenuePerMinute",
    "UsagePerDevice"
]

print("Summary Statistics for Engineered Features:\n")
print(df_clean[new_features].describe())

print("\nMissing Values:")
print(df_clean[new_features].isnull().sum())

print("\nInfinite Values:")
print(np.isinf(df_clean[new_features]).sum())

Summary Statistics for Engineered Features:

         TotalCalls  RevenuePerMinute  UsagePerDevice
count  51047.000000      51047.000000    51047.000000
mean     128.485244          0.362615      330.979664
std      139.348481          1.593759      323.152517
min        0.000000          0.000000        0.000000
25%       15.100000          0.082707      103.500000
50%       75.100000          0.131309      232.500000
75%      200.400000          0.233845      448.000000
max      474.600000         93.640000     1566.500000

Missing Values:
TotalCalls          0
RevenuePerMinute    0
UsagePerDevice      0
dtype: int64

Infinite Values:
TotalCalls          0
RevenuePerMinute    0
UsagePerDevice      0
dtype: int64


In [14]:
# ------------------------------------------
# Feature 3: Customer Tenure Group
# ------------------------------------------

# Define tenure intervals (in months)
tenure_bins = [0, 12, 24, 36, 48, 61]

tenure_labels = [
    "0-1 Year",
    "1-2 Years",
    "2-3 Years",
    "3-4 Years",
    "4-5 Years"
]

# Create TenureGroup
df_clean["TenureGroup"] = pd.cut(
    df_clean["MonthsInService"],
    bins=tenure_bins,
    labels=tenure_labels,
    include_lowest=True
)

# ------------------------------------------
# Validation
# ------------------------------------------

print("TenureGroup Distribution:\n")
print(df_clean["TenureGroup"].value_counts())

print("\nMissing Values:")
print(df_clean["TenureGroup"].isnull().sum())

TenureGroup Distribution:

TenureGroup
1-2 Years    21422
0-1 Year     16975
2-3 Years     9730
3-4 Years     2337
4-5 Years      583
Name: count, dtype: int64

Missing Values:
0


In [15]:

# ==========================================================
# Create Working Copy
# ==========================================================

df_fe = df_clean.copy()

# ==========================================================
# Clean Text Columns
# ==========================================================

cat_cols = df_fe.select_dtypes(include="object").columns

for col in cat_cols:
    df_fe[col] = (
        df_fe[col]
        .astype(str)
        .str.strip()
        .str.lower()
    )

# ==========================================================
# 1. Yes / No
# ==========================================================

cat_cols = df_fe.select_dtypes(include="object").columns

yes_no_cols = []

for col in cat_cols:
    values = set(df_fe[col].unique())

    if values == {"yes", "no"}:
        yes_no_cols.append(col)

print("Yes / No Columns:")
print(yes_no_cols)

for col in yes_no_cols:
    df_fe[col] = df_fe[col].map({
        "no": 0,
        "yes": 1
    })

# ==========================================================
# 2. Known / Unknown
# ==========================================================

cat_cols = df_fe.select_dtypes(include="object").columns

known_unknown_cols = []

for col in cat_cols:
    values = set(df_fe[col].unique())

    if values == {"known", "unknown"}:
        known_unknown_cols.append(col)

print("\nKnown / Unknown Columns:")
print(known_unknown_cols)

for col in known_unknown_cols:
    df_fe[col] = df_fe[col].map({
        "unknown": 0,
        "known": 1
    })

# ==========================================================
# 3. Credit Rating (Ordinal)
# ==========================================================

if "CreditRating" in df_fe.columns:

    credit_map = {
        "1-highest": 7,
        "2-high": 6,
        "3-good": 5,
        "4-medium": 4,
        "5-low": 3,
        "6-verylow": 2,
        "7-lowest": 1
    }

    df_fe["CreditRating"] = df_fe["CreditRating"].map(credit_map)

# ==========================================================
# 4. Tenure Group (Ordinal)
# ==========================================================

if "TenureGroup" in df_fe.columns:

    tenure_map = {
        "0-1 year": 0,
        "1-2 years": 1,
        "2-3 years": 2,
        "3-4 years": 3,
        "4-5 years": 4
    }

    df_fe["TenureGroup"] = (
        df_fe["TenureGroup"]
        .astype(str)
        .str.lower()
        .map(tenure_map)
    )

# ==========================================================
# 5. HandsetPrice
# ==========================================================

if "HandsetPrice" in df_fe.columns:

    df_fe["HandsetPrice"] = (
        df_fe["HandsetPrice"]
        .replace("unknown", np.nan)
    )

    df_fe["HandsetPrice"] = pd.to_numeric(
        df_fe["HandsetPrice"],
        errors="coerce"
    )

    df_fe["HandsetPrice"] = df_fe["HandsetPrice"].fillna(
        df_fe["HandsetPrice"].median()
    )

# ==========================================================
# 6. Remaining Categorical Columns
# ==========================================================

cat_cols = df_fe.select_dtypes(include="object").columns

print("\nRemaining Categorical Columns:")
print(list(cat_cols))

df_fe = pd.get_dummies(
    df_fe,
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

# ==========================================================
# 7. Scaling
# ==========================================================

num_cols = df_fe.select_dtypes(include=["int64", "float64"]).columns

num_cols = num_cols.drop("Churn", errors="ignore")

scaler = StandardScaler()

df_fe[num_cols] = scaler.fit_transform(df_fe[num_cols])

# ==========================================================
# Validation
# ==========================================================

print("\nFinal Shape:", df_fe.shape)

print("\nRemaining Missing Values:")
print(df_fe.isnull().sum().sum())

print("\nRemaining Object Columns:")
print(df_fe.select_dtypes(include="object").columns.tolist())

print("\nData Types:")
print(df_fe.dtypes.value_counts())

Yes / No Columns:
['Churn', 'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCreditCard', 'NewCellphoneUser', 'OwnsMotorcycle', 'MadeCallToRetentionTeam']

Known / Unknown Columns:
['Homeownership']

Remaining Categorical Columns:
['PrizmCode', 'Occupation', 'MaritalStatus', 'Market']

Final Shape: (51047, 123)

Remaining Missing Values:
0

Remaining Object Columns:
[]

Data Types:
int32      68
float64    54
int64       1
Name: count, dtype: int64


In [16]:
print(df_fe.head())
print(df_fe.describe().T.head())

   Churn  MonthlyRevenue  MonthlyMinutes  TotalRecurringCharge  \
0      1       -1.033942       -0.640973             -1.129992   
1      1       -1.265594       -1.122124             -1.362826   
2      0       -0.571298       -1.126728             -0.384922   
3      0        0.891977        1.875287              1.338054   
4      1       -1.260637       -1.145146             -1.362826   

   DirectorAssistedCalls  OverageMinutes  RoamingCalls  PercChangeMinutes  \
0              -0.405629       -0.697503     -0.622791          -0.999634   
1              -0.704436       -0.697503     -0.622791           0.030054   
2              -0.704436       -0.697503     -0.622791           0.043514   
3               0.777648       -0.697503     -0.622791           1.113583   
4              -0.704436       -0.697503     -0.622791           0.056974   

   PercChangeRevenues  DroppedCalls  BlockedCalls  UnansweredCalls  \
0           -1.693738     -0.799909     -0.570341        -0.741048   


In [17]:
print("="*50)
print("Task 3 Validation")
print("="*50)

print("Dataset Shape:", df_fe.shape)

print("\nMissing Values:")
print(df_fe.isnull().sum().sum())

print("\nDuplicate Records:")
print(df_fe.duplicated().sum())

print("\nObject Columns:")
print(df_fe.select_dtypes(include='object').columns.tolist())

print("\nTarget Distribution:")
print(df_fe["Churn"].value_counts())

Task 3 Validation
Dataset Shape: (51047, 123)

Missing Values:
0

Duplicate Records:
0

Object Columns:
[]

Target Distribution:
Churn
0    36336
1    14711
Name: count, dtype: int64


In [18]:
"""
Task 3 Summary:

During this task, new informative features were constructed to enhance the predictive power of the dataset. A new Market feature was extracted from the ServiceArea column, after which the original ServiceArea attribute was removed.

Additional behavioural features were engineered, including TotalCalls, RevenuePerMinute, and UsagePerDevice, to better represent customer usage patterns. The MonthsInService variable was also discretised into tenure groups to simplify customer lifecycle analysis.

Categorical variables were transformed into numerical representations using appropriate encoding techniques based on their characteristics. Binary variables were label encoded, ordinal variables such as CreditRating and TenureGroup were mapped according to their natural order, while nominal variables were converted using one-hot encoding. The mixed HandsetPrice feature was converted into a numerical attribute after handling unknown values.

Finally, all numerical predictors were standardised using StandardScaler to ensure a consistent feature scale for machine learning algorithms.

Validation confirmed that the transformed dataset contains 51,047 records and 124 features, with no missing values, no duplicate records, and no remaining categorical variables. The dataset is fully prepared for the modelling phase.
"""

'\nTask 3 Summary:\n\nDuring this task, new informative features were constructed to enhance the predictive power of the dataset. A new Market feature was extracted from the ServiceArea column, after which the original ServiceArea attribute was removed.\n\nAdditional behavioural features were engineered, including TotalCalls, RevenuePerMinute, and UsagePerDevice, to better represent customer usage patterns. The MonthsInService variable was also discretised into tenure groups to simplify customer lifecycle analysis.\n\nCategorical variables were transformed into numerical representations using appropriate encoding techniques based on their characteristics. Binary variables were label encoded, ordinal variables such as CreditRating and TenureGroup were mapped according to their natural order, while nominal variables were converted using one-hot encoding. The mixed HandsetPrice feature was converted into a numerical attribute after handling unknown values.\n\nFinally, all numerical predic

---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [19]:
# Since data is from a single source, no merging is required

---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [20]:
# ==========================================================
# Phase 3 - Task 5: Format Data
# ==========================================================

# Create final copy
df_final = df_fe.copy()

# ==========================================================
# Ensure Target Column is the Last Column
# ==========================================================

if "Churn" in df_final.columns:

    feature_cols = [col for col in df_final.columns if col != "Churn"]

    df_final = df_final[feature_cols + ["Churn"]]

# ==========================================================
# Final Validation
# ==========================================================

print("Final Dataset Shape:")
print(df_final.shape)

print("\nData Types:")
print(df_final.dtypes.value_counts())

print("\nMissing Values:")
print(df_final.isnull().sum().sum())

print("\nDuplicate Records:")
print(df_final.duplicated().sum())

print("\nTarget Position:")
print(df_final.columns[-1])

# ==========================================================
# Save Prepared Dataset
# ==========================================================

save_path = r"D:\Churn Prediction And Analysis Project\Cell2Cell Data\Cell2Cell_Prepared_Data.csv"

df_final.to_csv(save_path, index=False)

print(f"\nDataset saved successfully to:\n{save_path}")

Final Dataset Shape:
(51047, 123)

Data Types:
int32      68
float64    54
int64       1
Name: count, dtype: int64

Missing Values:
0

Duplicate Records:
0

Target Position:
Churn

Dataset saved successfully to:
D:\Churn Prediction And Analysis Project\Cell2Cell Data\Cell2Cell_Prepared_Data.csv
